# Laboratorul 7. Atenția: de la calculul de mână la un model real

**Transformers**

**Student:** Vladimir Russu

## 2. Pregătirea: descărcarea modelelor

Celula de mai jos descarcă o singură dată modelele folosite (câteva sute de MB fiecare). La rulările următoare vin din cache-ul Hugging Face.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

MODELE_TOKENIZER = ["gpt2", "bert-base-multilingual-cased",
                    "dumitrescustefan/bert-base-romanian-cased-v1"]
for nume in MODELE_TOKENIZER:
    AutoTokenizer.from_pretrained(nume)
AutoModelForCausalLM.from_pretrained("gpt2")
AutoModel.from_pretrained("dumitrescustefan/bert-base-romanian-cased-v1")
print("Gata: modelele sunt descărcate.")

Gata: modelele sunt descărcate.


## 3. Partea 1. Atenția în NumPy (35 min)

Implementăm mecanismul de atenție de la zero și îl verificăm pe exemplul rezolvat din lecție.

### Sarcina 1.1. Softmax stabil numeric

Formula din pasul 6 al lecției este $w_j = \dfrac{e^{s_j}}{\sum_m e^{s_m}}$. Scăderea maximului de pe fiecare rând nu schimbă rezultatul (numărătorul și numitorul se înmulțesc cu aceeași constantă $e^{-\max}$), dar evită depășirea numerică (`overflow`) pentru scoruri mari.

In [2]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

def softmax(s):
    e = np.exp(s - s.max(axis=-1, keepdims=True))   # scădem maximul de pe fiecare rând
    return e / e.sum(axis=-1, keepdims=True)         # împărțim la suma de pe fiecare rând

print(softmax(np.array([1.2, 2.8, 0.7])))

[0.152 0.755 0.092]


**Verificare:** trebuie să obținem `[0.152 0.755 0.092]`, ponderile din pasul 6 al lecției, nerotunjite.

Acum softmax „naiv”, fără scăderea maximului, pentru scorurile `[1000, 1001]`:

In [3]:
s = np.array([1000.0, 1001.0])

with np.errstate(over="ignore", invalid="ignore"):   # doar ca să nu umplem ecranul cu avertismente
    naiv = np.exp(s) / np.exp(s).sum()

print("exp(s) naiv:     ", np.exp(s[0]), np.exp(s[1]))
print("softmax naiv:    ", naiv)                      # varianta naivă
print("softmax stabil:  ", softmax(s))                # varianta noastră
print("softmax([0, 1]): ", softmax(np.array([0.0, 1.0])), "<- același rezultat, scorurile diferă doar cu o constantă")

exp(s) naiv:      inf inf
softmax naiv:     [nan nan]
softmax stabil:   [0.269 0.731]
softmax([0, 1]):  [0.269 0.731] <- același rezultat, scorurile diferă doar cu o constantă


Observație: `exp(1000)` depășește cel mai mare număr reprezentabil în `float64` (≈ 1.8·10³⁰⁸), deci devine `inf`, iar `inf / inf = nan`. Varianta stabilă calculează `exp(0)` și `exp(-1)`, numere perfect reprezentabile, și dă `[0.269 0.731]`. (Răspunsul complet este la întrebarea 1.)

### Sarcina 1.2. Funcția de atenție, verificată pe exemplul din curs

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [4]:
def attention(Q, K, V, mask=None):
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)                  # pasul 5: Q·Kᵀ / √d_k
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)     # pasul 14: pozițiile interzise primesc -∞
    weights = softmax(scores)                        # pasul 6
    return weights @ V, weights                      # pasul 7

# Datele din exemplul rezolvat (Lecția 5, după pasul 7)
q = np.array([[2, 1, 0, 1]])                         # q pentru „aprobat”
K = np.array([[0.5, 1.0, 0.3, 0.4],                  # k pentru „Banca”
              [2.0, 1.0, 0.5, 0.6],                  # k pentru „creditul”
              [0.2, 0.6, 0.9, 0.4]])                 # k pentru „clientului”
V = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [0.5, 0.5]])

z, w = attention(q, K, V)
print("scoruri brute q·kᵀ:", q @ K.T, " scalate cu √4 = 2:", q @ K.T / 2)
print("ponderi:", w)
print("z:", z)

assert np.allclose(w, [[0.152, 0.755, 0.092]], atol=1e-3)
assert np.allclose(z, [[0.199, 0.801]], atol=1e-3)
print("Verificare trecută.")

scoruri brute q·kᵀ: [[2.4 5.6 1.4]]  scalate cu √4 = 2: [[1.2 2.8 0.7]]
ponderi: [[0.152 0.755 0.092]]
z: [[0.199 0.801]]
Verificare trecută.


**Verificare:** ponderile sunt `[0.152 0.755 0.092]`, iar `z = [0.199 0.801]`. În lecție s-a obținut (0,195; 0,805) pentru că acolo s-au folosit ponderile rotunjite la două zecimale (0,15; 0,76; 0,09); rezultatul de aici este cel exact.

### Sarcina 1.3. Toți tokenii deodată, cu mască

O „propoziție” de 4 tokeni, fiecare un vector cu 8 componente, și trei matrice $W$ aleatoare. Seed-ul fix (42) asigură că toți obținem aceleași numere. Masca triunghiulară inferioară (pasul 14) permite fiecărui token să vadă doar tokenii de dinaintea lui și pe el însuși.

In [5]:
rng = np.random.default_rng(42)
X = rng.normal(size=(4, 8))                          # 4 tokeni × 8 componente
W_Q, W_K, W_V = (rng.normal(size=(8, 4)) for _ in range(3))

Q, K, V = X @ W_Q, X @ W_K, X @ W_V                  # pasul 4 din lecție

mask = np.tril(np.ones((4, 4), dtype=bool))         # True = tokenul poate fi văzut
print(mask)

Z, W = attention(Q, K, V, mask)
print(W)
print("Suma pe rânduri:", W.sum(axis=1))

assert np.allclose(W.sum(axis=1), 1)
assert np.allclose(np.triu(W, k=1), 0)
assert np.allclose(W[0], [1, 0, 0, 0])
assert np.allclose(W[1], [0.648, 0.352, 0, 0], atol=1e-3)
print("Verificare trecută.")

[[ True False False False]
 [ True  True False False]
 [ True  True  True False]
 [ True  True  True  True]]
[[1.    0.    0.    0.   ]
 [0.648 0.352 0.    0.   ]
 [0.491 0.145 0.363 0.   ]
 [0.285 0.341 0.281 0.093]]
Suma pe rânduri: [1. 1. 1. 1.]
Verificare trecută.


**Verificare:** suma pe fiecare rând este 1, deasupra diagonalei sunt doar zerouri, primul rând este `[1. 0. 0. 0.]`, iar al doilea `[0.648 0.352 0. 0.]`.

Pentru comparație, aceeași atenție **fără** mască (așa cum lucrează un encoder, care vede tot textul):

In [6]:
_, W_fara_masca = attention(Q, K, V)
print(W_fara_masca)
print("Suma pe rânduri:", W_fara_masca.sum(axis=1))

[[0.413 0.044 0.334 0.209]
 [0.241 0.131 0.332 0.297]
 [0.44  0.13  0.326 0.103]
 [0.285 0.341 0.281 0.093]]
Suma pe rânduri: [1. 1. 1. 1.]


### Sarcina 1.4. Multi-head attention cu două capete

Pasul 9 din lecție: fiecare cap calculează propria atenție cu propriile $W_Q, W_K, W_V$, rezultatele se concatenează și se combină cu $W_O$.

In [7]:
def multi_head(X, heads, W_O):
    # heads = listă de tupluri (W_Q, W_K, W_V), câte unul pe cap
    outs = [attention(X @ Wq, X @ Wk, X @ Wv)[0] for Wq, Wk, Wv in heads]
    concat = np.concatenate(outs, axis=-1)
    return concat @ W_O                              # matricea care combină capetele

heads = [tuple(rng.normal(size=(8, 4)) for _ in range(3)) for _ in range(2)]
W_O = rng.normal(size=(8, 8))

iesire = multi_head(X, heads, W_O)
print("ieșirea unui cap:", attention(X @ heads[0][0], X @ heads[0][1], X @ heads[0][2])[0].shape)
print("după concatenare:", np.concatenate([attention(X @ Wq, X @ Wk, X @ Wv)[0] for Wq, Wk, Wv in heads], axis=-1).shape)
print("după W_O:        ", iesire.shape)

assert iesire.shape == (4, 8)
print("Verificare trecută.")

ieșirea unui cap: (4, 4)
după concatenare: (4, 8)
după W_O:         (4, 8)
Verificare trecută.


**Verificare:** dimensiunea este `(4, 8)`: câte un vector de 8 componente pentru fiecare dintre cei 4 tokeni. Cele două capete dau câte 4 componente fiecare, iar după concatenare (4 × 8) și înmulțirea cu $W_O$ (8 × 8) ajungem la 8.

## 4. Partea 2. Tokenizarea (15 min)

Comparăm cum împart trei modele diferite același text: GPT-2 (antrenat mai ales pe engleză), BERT multilingv (104 limbi) și un BERT antrenat special pe limba română.

In [8]:
from transformers import AutoTokenizer

nume_modele = {
    "GPT-2": "gpt2",
    "BERT multilingv": "bert-base-multilingual-cased",
    "BERT românesc": "dumitrescustefan/bert-base-romanian-cased-v1",
}
tokenizere = {nume: AutoTokenizer.from_pretrained(m) for nume, m in nume_modele.items()}

fraze = ["Banca a aprobat creditul clientului.",
         "Împrumuturile restructurate au dobânzi variabile."]

numar_tokeni = {}
for nume, tok in tokenizere.items():
    for f in fraze:
        t = tok.tokenize(f)
        numar_tokeni[(nume, f)] = len(t)
        print(f"{nume:16} {len(t):3} {t}")

GPT-2             12 ['B', 'anca', 'Ġa', 'Ġa', 'pro', 'bat', 'Ġcredit', 'ul', 'Ġclient', 'ul', 'ui', '.']
GPT-2             20 ['Ã', 'İ', 'm', 'pr', 'um', 'ut', 'ur', 'ile', 'Ġrestruct', 'urate', 'Ġau', 'Ġdo', 'b', 'Ã¢', 'n', 'zi', 'Ġvari', 'ab', 'ile', '.']
BERT multilingv   10 ['Banca', 'a', 'ap', '##ro', '##bat', 'credit', '##ul', 'client', '##ului', '.']
BERT multilingv   15 ['Î', '##mp', '##rumu', '##turi', '##le', 'rest', '##ru', '##ctura', '##te', 'au', 'dob', '##ân', '##zi', 'variabile', '.']
BERT românesc      6 ['Banca', 'a', 'aprobat', 'creditul', 'clientului', '.']
BERT românesc      8 ['Împrum', '##uturile', 'restruct', '##urate', 'au', 'dobânzi', 'variabile', '.']


GPT-2 lucrează la nivel de octeți, așa că literele românești (ă, î, ș, ț) apar ca simboluri ciudate (de exemplu `Ä`). Este normal. Simbolul `Ġ` marchează un spațiu înaintea tokenului, iar `##` la BERT marchează continuarea unui cuvânt.

### Sarcina 2.1. Tabelul cu numărul de tokeni

In [9]:
print(f"{'Model':16} {'Fraza 1':>8} {'Fraza 2':>8}")
for nume in nume_modele:
    print(f"{nume:16} {numar_tokeni[(nume, fraze[0])]:8} {numar_tokeni[(nume, fraze[1])]:8}")

Model             Fraza 1  Fraza 2
GPT-2                  12       20
BERT multilingv        10       15
BERT românesc           6        8


### Sarcina 2.2. Cât „costă” limba română?

Numărăm cu GPT-2 tokenii pentru aceeași frază în engleză și calculăm raportul. Serviciile API se plătesc per token, iar fereastra de context se măsoară tot în tokeni.

In [10]:
gpt2 = tokenizere["GPT-2"]
n_ro = len(gpt2.tokenize("Banca a aprobat creditul clientului."))
n_en = len(gpt2.tokenize("The bank approved the client's loan."))
print(n_ro, n_en, round(n_ro / n_en, 2))

print("EN:", gpt2.tokenize("The bank approved the client's loan."))

pret_per_milion = 1.0   # dolari per 1 000 000 de tokeni
fraze_prelucrate = 1_000_000
print(f"1 milion de fraze în română:  {n_ro * fraze_prelucrate / 1e6 * pret_per_milion:.2f} $")
print(f"1 milion de fraze în engleză: {n_en * fraze_prelucrate / 1e6 * pret_per_milion:.2f} $")

12 8 1.5
EN: ['The', 'Ġbank', 'Ġapproved', 'Ġthe', 'Ġclient', "'s", 'Ġloan', '.']
1 milion de fraze în română:  12.00 $
1 milion de fraze în engleză: 8.00 $


### Sarcina 2.3. Diacriticele cu virgulă și cu sedilă

În textele românești apar două variante ale acelorași litere: ș, ț (cu virgulă, forma corectă) și ş, ţ (cu sedilă, o formă veche, încă frecventă). Pentru calculator sunt caractere diferite.

In [11]:
ro = tokenizere["BERT românesc"]
t_virgula = ro.tokenize("știință și țară")     # ș, ț cu virgulă (U+0219, U+021B)
t_sedila  = ro.tokenize("ştiinţă şi ţară")     # ş, ţ cu sedilă  (U+015F, U+0163)
print(len(t_virgula), t_virgula)
print(len(t_sedila),  t_sedila)

print("\nCodurile Unicode:", [f"{c} U+{ord(c):04X}" for c in "șț"], "vs", [f"{c} U+{ord(c):04X}" for c in "şţ"])

# Normalizarea pe care ar trebui să o facem înainte de a da textul unui model:
NORMALIZARE = str.maketrans("şţŞŢ", "șțȘȚ")
print("\nDupă normalizare:", ro.tokenize("ştiinţă şi ţară".translate(NORMALIZARE)))

3 ['știință', 'și', 'țară']
3 ['[UNK]', '[UNK]', '[UNK]']

Codurile Unicode: ['ș U+0219', 'ț U+021B'] vs ['ş U+015F', 'ţ U+0163']

După normalizare: ['știință', 'și', 'țară']


## 5. Partea 3. Generarea textului (30 min)

Folosim GPT-2, un model de tip „doar decoder” (lecția, secțiunea 6). Lucrăm în engleză, pentru că GPT-2 a fost antrenat pe texte în engleză.

### Sarcina 3.1. Probabilitățile pentru următorul token (pasul 13)

In [12]:
import torch
from transformers import AutoModelForCausalLM

tok = tokenizere["GPT-2"]
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

def top10(model, tok, text):
    ids = tok(text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(ids).logits[0, -1]            # scorurile pentru ULTIMA poziție
    print("Dimensiunea vectorului logits:", logits.shape, "| mărimea vocabularului:", len(tok))
    probs = torch.softmax(logits, dim=-1)
    print("Suma probabilităților:", probs.sum().item())
    top = torch.topk(probs, 10)
    for p, i in zip(top.values, top.indices):
        print(f"{tok.decode(int(i))!r:16} {p.item():.3f}")
    return top

top_gpt2 = top10(model, tok, "The bank approved the")

Dimensiunea vectorului logits: torch.Size([50257]) | mărimea vocabularului: 50257
Suma probabilităților: 1.0000050067901611
' loan'          0.066
' deal'          0.066
' sale'          0.037
' transaction'   0.033
' plan'          0.025
' purchase'      0.024
' new'           0.019
' $'             0.019
' move'          0.016
' settlement'    0.012


### Sarcina 3.2. Bucla de generare, scrisă de mână (pasul 17)

În loc de `model.generate()`, scriem singuri bucla: **prezice → alege → adaugă → repetă**. Temperatura împarte scorurile înainte de Softmax.

In [13]:
def genereaza(model, tok, text, n_tokeni=25, temperatura=1.0, greedy=False):
    ids = tok(text, return_tensors="pt").input_ids
    for _ in range(n_tokeni):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]                 # 1. prezice
        if greedy:
            urm = torch.argmax(logits)                         # 2. alege: tokenul cu scor maxim
        else:
            probs = torch.softmax(logits / temperatura, dim=-1)   #    temperatura împarte scorurile
            urm = torch.multinomial(probs, 1)[0]               #    extragere aleatoare
        ids = torch.cat([ids, urm.view(1, 1)], dim=1)          # 3. adaugă tokenul
    return tok.decode(ids[0])                                  # 4. repetă

torch.manual_seed(5)   # ca extragerile aleatoare să fie reproductibile la re-rulare

text = "The bank approved the"
print("GREEDY:", genereaza(model, tok, text, greedy=True))
print("GREEDY:", genereaza(model, tok, text, greedy=True), "<- a doua rulare, identică")
print()
for T in [0.2, 0.7, 1.5]:
    for incercare in range(2):
        print(f"T = {T}:", genereaza(model, tok, text, temperatura=T))
    print()

GREEDY: The bank approved the loan in May, but the bank has not yet approved the loan.

The bank said it will not be able to


GREEDY: The bank approved the loan in May, but the bank has not yet approved the loan.

The bank said it will not be able to <- a doua rulare, identică



T = 0.2: The bank approved the loan in September.

The bank said it will continue to lend to the bank until the loan is repaid.




T = 0.2: The bank approved the loan in September, but the bank has not yet received a response from the government.

The government has also been criticized



T = 0.7: The bank approved the deal in March 2016, when the government announced the deal after the European Financial Stability Facility (EFSF) was approved by


T = 0.7: The bank approved the fund as a "proceeds from the sale of shares in the company, which is within the regulatory framework of its Regulation



T = 1.5: The bank approved the liability plan Saturday, along with developed pledges designed for victims squirmy victims climbing by half last year and cabanos Human disclosed


T = 1.5: The bank approved the photos as currency in 2007 on nationwide sank currencies dur''m off sooner.Quantom purchased to feeding subscribers 144 more ultralis



Efectul temperaturii asupra distribuției, pe primii 10 tokeni (același `logits`, împărțit la T):

In [14]:
ids = tok(text, return_tensors="pt").input_ids
with torch.no_grad():
    logits = model(ids).logits[0, -1]

print(f"{'token':14}" + "".join(f"{'T=' + str(T):>9}" for T in [0.2, 0.7, 1.0, 1.5]))
for i in top_gpt2.indices[:10]:
    linie = f"{tok.decode(int(i))!r:14}"
    for T in [0.2, 0.7, 1.0, 1.5]:
        p = torch.softmax(logits / T, dim=-1)[i].item()
        linie += f"{p:9.3f}"
    print(linie)
for T in [0.2, 0.7, 1.0, 1.5]:
    p = torch.softmax(logits / T, dim=-1)
    print(f"T = {T}: masa de probabilitate a primilor 10 tokeni = {p[top_gpt2.indices].sum().item():.3f}")

token             T=0.2    T=0.7    T=1.0    T=1.5
' loan'           0.482    0.175    0.066    0.013
' deal'           0.467    0.174    0.066    0.013
' sale'           0.027    0.077    0.037    0.009
' transaction'    0.014    0.064    0.033    0.008
' plan'           0.004    0.045    0.025    0.007
' purchase'       0.003    0.041    0.024    0.006
' new'            0.001    0.029    0.019    0.006
' $'              0.001    0.029    0.019    0.006
' move'           0.000    0.022    0.016    0.005
' settlement'     0.000    0.016    0.012    0.004
T = 0.2: masa de probabilitate a primilor 10 tokeni = 1.000
T = 0.7: masa de probabilitate a primilor 10 tokeni = 0.672
T = 1.0: masa de probabilitate a primilor 10 tokeni = 0.316
T = 1.5: masa de probabilitate a primilor 10 tokeni = 0.076


#### Tabelul de observații

| Setare | Ce am observat |
|---|---|
| Greedy | Determinist: de fiecare dată exact același text, pentru că se alege mereu tokenul cu probabilitatea maximă. Textul este gramatical, dar plat și tinde să intre în bucle (repetă aceeași construcție). |
| T = 0,2 | Aproape identic cu greedy: distribuția devine foarte „ascuțită”, primul token ia aproape toată masa de probabilitate. Cele două încercări diferă puțin sau deloc; textul este coerent, dar repetitiv. |
| T = 0,7 | Compromisul bun: textul rămâne coerent și pe subiect (bancă, împrumut, bani), dar cele două încercări sunt clar diferite. |
| T = 1,5 | Variat, dar din ce în ce mai incoerent: apar cuvinte rare, schimbări bruște de subiect, uneori fragmente fără sens. Distribuția este aproape plată, deci tokenii improbabili sunt aleși des. |

### Sarcina 3.3. (opțional) Un model pentru limba română

Repetăm sarcinile 3.1 și 3.2 cu RoGPT2, un GPT-2 antrenat pe texte în română, și textul „Banca a aprobat”. RoGPT2-base este un model mic (2021); româna generată este imperfectă, dar ne interesează mecanismul.

In [15]:
tok_ro_gpt = AutoTokenizer.from_pretrained("readerbench/RoGPT2-base")
model_ro_gpt = AutoModelForCausalLM.from_pretrained("readerbench/RoGPT2-base")
model_ro_gpt.eval()

top10(model_ro_gpt, tok_ro_gpt, "Banca a aprobat")
print()
torch.manual_seed(5)
print(genereaza(model_ro_gpt, tok_ro_gpt, "Banca a aprobat", temperatura=0.7))

Dimensiunea vectorului logits: torch.Size([50257]) | mărimea vocabularului: 50257
Suma probabilităților: 1.0000048875808716
' în'            0.108
' un'            0.075
','              0.060
' o'             0.050
'-'              0.049
' finanțarea'    0.013
' planul'        0.013
' pentru'        0.013
' la'            0.012
' și'            0.010



Banca a aprobat în iulie 2008 o emisiune de monede metalice la cerere, în valoare de 1,2 miliarde de lei. Banca a cerut alte


## 6. Partea 4 (opțională). Embeddings contextuale

Verificăm exemplul din lecție (pașii 2 și 8): același cuvânt „bancă” primește vectori diferiți în funcție de context. Folosim BERT-ul românesc, care are 12 blocuri. Stratul 0 este ieșirea din pasul 3 (embedding + poziție), iar stratul 12 ieșirea ultimului bloc.

### Sarcina 4.1. Vectorul lui „bancă” în patru propoziții

In [16]:
import torch.nn.functional as Fn

nume_ro = "dumitrescustefan/bert-base-romanian-cased-v1"
tok_bert = AutoTokenizer.from_pretrained(nume_ro)
bert = AutoModel.from_pretrained(nume_ro, output_hidden_states=True, attn_implementation="eager")
bert.eval()

def vector_cuvant(fraza, cuvant, strat):
    enc = tok_bert(fraza, return_tensors="pt")
    with torch.no_grad():
        out = bert(**enc)
    tokens = tok_bert.convert_ids_to_tokens(enc.input_ids[0])
    parti = tok_bert.tokenize(cuvant)                # cuvântul poate avea mai mulți tokeni
    for i in range(len(tokens) - len(parti) + 1):
        if tokens[i:i + len(parti)] == parti:
            return out.hidden_states[strat][0, i:i + len(parti)].mean(dim=0)
    raise ValueError(f"Nu am găsit „{cuvant}” în: {fraza}")

fraze_banca = ["Am deschis un cont la bancă.",             # 1: sens financiar
               "Am luat un credit de la bancă.",           # 2: sens financiar
               "M-am așezat pe o bancă în parc.",          # 3: loc de șezut
               "Bătrânul stă pe bancă lângă fântână."]     # 4: loc de șezut

similaritati = {}
for strat in [0, 12]:
    vecs = torch.stack([vector_cuvant(f, "bancă", strat) for f in fraze_banca])
    sim = Fn.cosine_similarity(vecs.unsqueeze(1), vecs.unsqueeze(0), dim=-1)
    similaritati[strat] = sim
    print(f"Stratul {strat}:")
    print(sim.numpy().round(3))

print()
for strat in [0, 12]:
    s = similaritati[strat]
    print(f"Stratul {strat:2}: sim(1,2) = {s[0,1]:.3f}  sim(1,3) = {s[0,2]:.3f}  sim(1,4) = {s[0,3]:.3f}  sim(3,4) = {s[2,3]:.3f}")
s12 = similaritati[12]
financiar = s12[0, 1].item()
mixt = torch.stack([s12[0, 2], s12[0, 3], s12[1, 2], s12[1, 3]]).mean().item()
print(f"\nStratul 12: media perechilor financiar–financiar = {financiar:.3f}, media perechilor financiar–șezut = {mixt:.3f}")
print("Propozițiile 1 și 2 sunt mai apropiate între ele decât de 3 și 4:", financiar > mixt)

Stratul 0:
[[1.    0.971 0.971 0.961]
 [0.971 1.    1.    0.954]
 [0.971 1.    1.    0.954]
 [0.961 0.954 0.954 1.   ]]


Stratul 12:
[[1.    0.953 0.659 0.594]
 [0.953 1.    0.625 0.56 ]
 [0.659 0.625 1.    0.832]
 [0.594 0.56  0.832 1.   ]]

Stratul  0: sim(1,2) = 0.971  sim(1,3) = 0.971  sim(1,4) = 0.961  sim(3,4) = 0.954
Stratul 12: sim(1,2) = 0.953  sim(1,3) = 0.659  sim(1,4) = 0.594  sim(3,4) = 0.832

Stratul 12: media perechilor financiar–financiar = 0.953, media perechilor financiar–șezut = 0.609
Propozițiile 1 și 2 sunt mai apropiate între ele decât de 3 și 4: True


### Sarcina 4.2. Vizualizarea atenției cu bertviz

Vizualizarea este interactivă (JavaScript): se afișează în Jupyter Notebook / JupyterLab sau în browser. Alegeți stratul din lista de sus și dați clic pe un token ca să vedeți la cine se uită.

In [17]:
from bertviz import head_view

bert_att = AutoModel.from_pretrained(nume_ro, output_attentions=True, attn_implementation="eager")
bert_att.eval()
enc = tok_bert("Banca a aprobat creditul clientului.", return_tensors="pt")
with torch.no_grad():
    att = bert_att(**enc).attentions
tokens_viz = tok_bert.convert_ids_to_tokens(enc.input_ids[0])
head_view(att, tokens_viz)

<IPython.core.display.Javascript object>

Ca să nu depindem de vizualizarea interactivă, căutăm și numeric capetele în care „aprobat” se uită mult la „creditul”:

In [18]:
i_aprobat = tokens_viz.index("aprobat")
i_creditul = tokens_viz.index("creditul")
print("tokeni:", tokens_viz)

rezultate = []
for strat, a in enumerate(att):                    # 12 straturi
    for cap in range(a.shape[1]):                  # 12 capete
        rezultate.append((a[0, cap, i_aprobat, i_creditul].item(), strat, cap))
rezultate.sort(reverse=True)
print("\nCapetele în care „aprobat” se uită cel mai mult la „creditul” (pondere, strat, cap):")
for pondere, strat, cap in rezultate[:5]:
    print(f"  {pondere:.3f}   stratul {strat + 1:2}, capul {cap + 1:2}")

pondere, strat, cap = rezultate[0]
print(f"\nStratul {strat + 1}, capul {cap + 1}: la cine se uită fiecare token")
a = att[strat][0, cap]
for i, t in enumerate(tokens_viz):
    j = int(a[i].argmax())
    print(f"  {t:12} -> {tokens_viz[j]:12} ({a[i, j]:.2f})")

tokeni: ['[CLS]', 'Banca', 'a', 'aprobat', 'creditul', 'clientului', '.', '[SEP]']

Capetele în care „aprobat” se uită cel mai mult la „creditul” (pondere, strat, cap):
  0.995   stratul  2, capul  4
  0.947   stratul  3, capul 11
  0.891   stratul  8, capul 12
  0.560   stratul  5, capul  8
  0.464   stratul 10, capul  8

Stratul 2, capul 4: la cine se uită fiecare token
  [CLS]        -> [CLS]        (0.99)
  Banca        -> a            (1.00)
  a            -> aprobat      (1.00)
  aprobat      -> creditul     (0.99)
  creditul     -> clientului   (1.00)
  clientului   -> .            (1.00)
  .            -> [SEP]        (1.00)
  [SEP]        -> [SEP]        (0.98)


## 7. Întrebări (răspunsuri cu numerele obținute)

### 1. (Sarcina 1.1) Softmax naiv vs. stabil pentru `[1000, 1001]`

Softmax naiv a dat `[nan nan]`, cu avertismente de `overflow` și `invalid value`. Varianta noastră a dat `[0.269 0.731]`.

Diferența apare pentru că `exp(1000)` și `exp(1001)` depășesc cel mai mare număr reprezentabil în `float64` (≈ 1,8·10³⁰⁸) și devin `inf`; apoi `inf / (inf + inf) = nan`. Scăderea maximului transformă scorurile în `[-1, 0]`, iar `exp(-1) = 0.368` și `exp(0) = 1` sunt numere obișnuite.

Scăderea maximului nu schimbă rezultatul corect pentru că $\dfrac{e^{s_j - c}}{\sum_m e^{s_m - c}} = \dfrac{e^{-c}\,e^{s_j}}{e^{-c}\sum_m e^{s_m}} = \dfrac{e^{s_j}}{\sum_m e^{s_m}}$: constanta $e^{-c}$ se simplifică. Softmax depinde doar de *diferențele* dintre scoruri (1001 − 1000 = 1), de aceea `softmax([0, 1])` dă exact aceleași `[0.269 0.731]`.

### 2. (Sarcina 1.3) Matricea de ponderi mascate

Primul rând este `[1, 0, 0, 0]` pentru că primul token nu are niciun token înaintea lui: masca pune `-inf` pe pozițiile 2, 3 și 4, `exp(-inf) = 0`, iar singurul scor rămas primește toată probabilitatea, adică exact 1 (Softmax pe un singur element este întotdeauna 1, indiferent de valoarea scorului).

Al treilea token „vede” tokenii 1, 2 și 3 (pe el însuși și pe cei doi de dinainte), cu ponderile `[0.491 0.145 0.363 0.]`; tokenul 4 este în „viitor” și are pondere exact 0. Așa funcționează decoder-ul (pasul 14): la antrenare toate pozițiile se calculează în paralel, dar fiecare poziție vede doar trecutul, ca la generare.

### 3. (Sarcina 2.2) Cât „costă” limba română la GPT-2

Fraza în română are **12** tokeni, cea în engleză **8**, deci raportul este **1,5** (româna este de 1,5 ori mai „scumpă”). La 1 $ per milion de tokeni, un milion de fraze ca aceasta ar costa **12 $** în română și **8 $** în engleză.

Explicația: vocabularul GPT-2 a fost construit pe text aproape exclusiv englezesc, așa că cuvintele englezești comune sunt un singur token, iar cuvintele românești (mai ales cele cu diacritice, codificate pe 2 octeți în UTF-8) sunt sparte în multe fragmente mici. Aceeași frază consumă așadar mai mulți tokeni din fereastra de context și costă mai mult la un API plătit per token.

### 4. (Sarcina 2.3) „știință” vs. „ştiinţă”

Varianta cu virgulă (ș, ț: U+0219, U+021B) a fost împărțită în **3** tokeni (`['știință', 'și', 'țară']`, fiecare cuvânt un token întreg), iar cea cu sedilă (ş, ţ: U+015F, U+0163) tot în **3** tokeni, dar toți trei sunt `[UNK]`: `['[UNK]', '[UNK]', '[UNK]']`. Pentru model sunt litere complet diferite; literele cu sedilă nu există deloc în vocabularul BERT-ului românesc (nici măcar ca litere separate), deci cuvintele cad pe tokenul necunoscut `[UNK]` și tot sensul lor se pierde.

Înainte de a da un set de texte românești unui model, textele trebuie **normalizate**: toate ş/ţ cu sedilă înlocuite cu ș/ț cu virgulă (de exemplu `str.translate`, ca în celula de mai sus, sau normalizare Unicode), plus curățarea altor variante (diacritice lipsă, encodări greșite). Așa același cuvânt primește mereu aceiași tokeni și același embedding.

### 5. (Sarcina 3.1) Vectorul logits la GPT-2

Vectorul `logits` are dimensiunea **50257**: câte un scor pentru fiecare token din vocabularul GPT-2 (50 000 de fuziuni BPE + 256 de octeți de bază + tokenul special `<|endoftext|>`). Este ieșirea stratului Linear din pasul 13; după Softmax devine o distribuție de probabilitate peste întregul vocabular, cu suma 1 (am obținut 1.000005; abaterea este eroare de rotunjire în `float32`).

Primii trei tokeni propuși după „The bank approved the”: ` loan` (0,066), ` deal` (0,066) și ` sale` (0,037); urmează ` transaction` (0,033) și ` plan` (0,025). Toate sunt continuări plauzibile în context bancar, iar primele două au practic aceeași probabilitate, deci chiar și greedy alege între ele „la limită”.

### 6. (Sarcina 3.2) Greedy și temperatura

Greedy produce mereu același text pentru că nu există nicio sursă de aleatoriu: la fiecare pas se ia `argmax(logits)`, iar modelul este o funcție deterministă a textului de intrare. Același prefix dă aceleași logits, deci același token, deci același text; de aceea cele două rulări greedy sunt identice. Greedy tinde și să se repete (ciclu „the bank … the bank …”), pentru că odată intrat într-o buclă, aceeași intrare dă aceeași continuare.

Temperatura împarte scorurile înainte de Softmax: $w_j = \dfrac{e^{s_j/T}}{\sum_m e^{s_m/T}}$. Când $T$ scade (0,2), diferențele dintre scoruri sunt amplificate de 5 ori, distribuția devine foarte ascuțită și aproape toată masa se mută pe tokenul maxim: textul devine aproape identic cu greedy, coerent dar repetitiv. Când $T$ crește (1,5), diferențele se micșorează, distribuția se aplatizează și tokenii improbabili ajung să fie extrași des: textul devine variat, dar tot mai incoerent. Tabelul cu probabilitățile primilor 10 tokeni la T = 0,2 / 0,7 / 1,0 / 1,5 arată exact acest efect (masa primilor 10 tokeni scade pe măsură ce T crește). La $T \to 0$ Softmax devine `argmax` (greedy), iar la $T \to \infty$ devine distribuție uniformă.

### 7. (Partea 4, opțional) Similaritățile lui „bancă”

| Perechea | Stratul 0 | Stratul 12 |
|---|---|---|
| 1 și 2 (cont la bancă / credit de la bancă): același sens | 0,971 | 0,953 |
| 1 și 3 (cont la bancă / bancă în parc): sensuri diferite | 0,971 | 0,659 |

La stratul 0 similaritățile sunt foarte mari pentru orice pereche: „bancă” are același embedding de token (pasul 2) în toate propozițiile, vectorii diferă doar prin codificarea poziției (pasul 3), deci stratul 0 **nu** distinge sensurile: perechea 1–2 și perechea 1–3 au exact aceeași similaritate (0,971), iar propozițiile 2 și 3 au chiar 1,000, pentru că „bancă” stă pe aceeași poziție în amândouă. La stratul 12, după 12 blocuri de atenție (pasul 8), vectorul lui „bancă” a absorbit contextul: perechea cu același sens (1–2, financiar) este mai apropiată decât perechea cu sensuri diferite (1–3). Diferența este mult mai clară decât ne avertiza fișa: 0,953 pentru 1–2 față de 0,659 pentru 1–3 (media perechilor financiar–șezut este 0,609, iar perechea 3–4, ambele cu sensul „loc de șezut”, are 0,832). Direcția este cea așteptată: embeddings-urile contextuale separă sensurile cuvântului, cele statice nu.